# 01 — Data Audit

Grain, schema, and the question that matters most for a demand panel: **is a zero a real zero?**

A zero can mean the product was on shelf and nobody bought it, or that it did not exist yet, or that it was delisted. Training on all three as the same thing teaches the model that demand collapses at the start and end of a product's life — an artefact of the panel's shape.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

## Grain and schema

In [ ]:
from src.data.schema import DATA_DICTIONARY
import pandas as pd
print('grain:', cfg['data']['grain'], '| target:', cfg.target)
pd.DataFrame([{'column': s.name, 'kind': s.kind, 'source': s.source,
               'description': s.description, 'notes': s.notes}
              for s in DATA_DICTIONARY])

## Structural integrity and the zero taxonomy

In [ ]:
from src.data.audit import zero_taxonomy, lifecycle, calendar_gaps, spikes
print('duplicate (series,date):', int(panel.duplicated(['series_id','date']).sum()))
print('series with internal gaps:', calendar_gaps(panel))
print('negative demand:', int((panel.demand < 0).sum()))
zero_taxonomy(panel)

In [ ]:
lifecycle(panel)

In [ ]:
spikes(panel)

## Intermittency: the fact that shapes every later choice

In [ ]:
from src.data.loader import demand_stats, classify_demand
st = demand_stats(panel); st['demand_class'] = classify_demand(st)
print(st.demand_class.value_counts())
st[['ADI','CV2','zero_share','mean_demand']].describe().round(3)

## The full audit, including the executable leakage probe

The probe rebuilds features after overwriting every actual past the forecast origin. Any feature that changes was reading the future.

In [ ]:
from src.data import audit
res = audit.run(cfg)

In [ ]:
res.leakage_checks